<a href="https://colab.research.google.com/github/halitcoskun/project/blob/main/MHRS_Llama3_Egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
# @title Varsayılan başlık metni
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Veri setini doğrudan Hugging Face repondaki JSON dosyasından çekiyoruz
# (HF hub üzerinden json dosyasını okumak için "json" tipini ve hf:// linkini kullanmak en güvenli yoldur)
dataset = load_dataset("json", data_files="hf://datasets/halitcoskun/mhrs-randevu-veri-seti/dataset.json", split="train")

# 2. Llama-3'ün Chat formatını sisteme tanıtıyoruz
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

# 3. Verimizi Llama 3'ün anlayacağı yapıya çeviriyoruz
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 160,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.705 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

In [ ]:
from unsloth import FastLanguageModel

# Modeli çıkarım moduna al
FastLanguageModel.for_inference(model)

# Eğitimde kullandığın mesaj yapısının aynısını kuruyoruz
messages = [
    {"role": "system", "content": "Sen otonom bir MHRS sesli asistanısın. Kullanıcının talebini analiz et ve JSON çıktısı üret. Bilgi eksikse 'ask_user' fonksiyonunu kullan."},
    {"role": "user", "content": "Merdivenleri çıkarken nefesim kesiliyor, çabuk yoruluyorum. Randevu alır mısın bana?"}
]

# Chat şablonunu uygula (add_generation_prompt=True asistanın cevabını başlatır)
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Yanıtı oluştur
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 128, # JSON çıktısı için yeterli uzunluk
    use_cache = True,
    temperature = 0.1,    # Daha tutarlı JSON çıktıları için düşük tutulmalı
    top_p = 0.9
)

# Yanıtı çöz ve sadece asistanın yeni ürettiği kısmı göster
decoded_output = tokenizer.batch_decode(outputs)
print(decoded_output[0])

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

In [ ]:
model.save_pretrained_gguf("mhrs_asistan", tokenizer, quantization_method = "q4_k_m")